# Module 6: Explore Inventory Signals and Review Pilot Cases

目标：用 Module 5 的试点配对表，观察本期库存增长差与下一年度经营表现是否存在值得进一步研究的关系。
与前面相同：每格先讲构建思路，再给代码；教学说明用中文，代码、注释、图表及输出字段用英文。
采用 pandas 和 matplotlib，普通循环，不使用自定义分析函数、机器学习或复杂回归。

输入：pilot_next_year_pairs.csv 和 Module 5 metadata。
输出：公司概要、分组描述、敏感性检查、待审核个案、两张 PNG 图片与 metadata。
本模块可以接在现有 Notebook 后，也可以单独运行；项目路径需与之前一致。

当前只有 3 家公司、43 个公司年度配对。同一公司历年记录存在联系，这不是 43 家独立公司。
各公司成本范围不同，因此按公司分别观察变化，不把原始毛利率水平直接进行优劣排名。

这是探索性年度关联研究。年报提交时下一财年已经开始，历史口径尚未全部审核；本模块不做统计显著性、因果识别、投资建议或收益回测。
数值不支持原假设也有研究价值；不要为得到预期结果反复改阈值、删年份或删公司。


## Cell 1: Set Up the Plotting Tools and Directories

新增 matplotlib 用于画图。%pip install matplotlib 安装到当前 Notebook 的 Kernel；已有时通常只提示已安装。
%matplotlib inline 让图显示在 Notebook 的输出区；两行百分号开头的语句是 Jupyter 指令，不是普通 Python，不能直接复制到 .py 脚本。
若安装后导入失败，重启 Kernel 后从本模块第一格重跑。

PROCESSED_DIR 仍读取之前的整理数据。FIGURE_DIR 用于保存可以插入报告的 PNG 图片。
mkdir 创建目录，parents=True 允许创建父目录，exist_ok=True 允许目录已存在。
图表字号适度统一，不改变财务数据。


In [ ]:
%pip install matplotlib

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

%matplotlib inline

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["font.size"] = 10
print("Project directory:", PROJECT_ROOT)


## Cell 2: Read the Paired Sample and Set Display Units

读取 Module 5 筛选后的合格配对，不重新删除观察值。
dtype 恢复可缺失布尔类型；parse_dates 恢复需要展示的日期。
metadata 保留上一模块的来源与研究边界。

原表用小数保存比率和变化。本模块新增 _pp 列，把三个变化指标乘 100，图上的单位变为百分点。
例如 inventory_sales_gap=0.10，对应 inventory_gap_pp=10.0，即存货增长高出收入增长 10 个百分点。
这种转换只为展示，不改变原始列，也不把比率差误解释为金额增长。
source_caption 标明图表来源、样本量和本期年度终点范围。


In [ ]:
analysis = pd.read_csv(
    PROCESSED_DIR / "pilot_next_year_pairs.csv",
    parse_dates=["period_end", "next_period_end", "signal_available_date"],
    dtype={"gross_margin_declined": "boolean", "ocf_margin_declined": "boolean"},
)
module5_metadata = json.loads(
    (PROCESSED_DIR / "pilot_module5_metadata.json").read_text(encoding="utf-8")
)
analysis["inventory_gap_pp"] = analysis["inventory_sales_gap"] * 100
analysis["next_gross_margin_change_pp"] = analysis["next_gross_margin_change"] * 100
analysis["next_ocf_margin_change_pp"] = analysis["next_ocf_margin_change"] * 100
plot_columns = ["inventory_gap_pp", "next_gross_margin_change_pp", "next_ocf_margin_change_pp"]
source_caption = (
    f"Source: cached SEC Company Facts | {len(analysis)} annual pairs | "
    f"Current period ends: {analysis['period_end'].min().date()} to {analysis['period_end'].max().date()}"
)
print("Pilot paired rows:", len(analysis))
display(analysis[plot_columns].describe())


## Cell 3: Summarize Each Company

.groupby("ticker") 分别汇总公司，避免把不同公司直接混成一个平均值。
.agg 的写法是 新列名=(输入列名, 汇总方法)。size 计算行数；mean 计算平均值。
布尔值的均值可以解释为下降的观察比例，因为 True 对应 1，False 对应 0。
例如 gross_margin_decline_rate=0.60 表示该公司配对中 60% 的下一年度毛利率下降，不是未来下降概率的模型估计。

当前预计 AEO 13 行、ANF 15 行、URBN 15 行。
这份表是样本描述。它不证明库存指标有预测能力，也不解释平均变化的原因。


In [ ]:
company_summary = analysis.groupby("ticker").agg(
    paired_rows=("ticker", "size"),
    mean_inventory_gap_pp=("inventory_gap_pp", "mean"),
    mean_next_gross_margin_change_pp=("next_gross_margin_change_pp", "mean"),
    mean_next_ocf_margin_change_pp=("next_ocf_margin_change_pp", "mean"),
    gross_margin_decline_rate=("gross_margin_declined", "mean"),
    ocf_margin_decline_rate=("ocf_margin_declined", "mean"),
)
display(company_summary.round(3))


## Cell 4: Plot Inventory Gaps against Next-Year Changes

一张图包含两个面板：库存增长差与下一年度毛利率变化，以及与下一年度经营现金流收入比变化。
plt.subplots(1, 2) 创建一行两张图；zip 把每个坐标区、目标列和标题依次配对。
内部循环每次画一家公司的点，颜色与图例用于识别公司。

横轴为本期库存增长差（百分点）；纵轴为下一年度比率变化（百分点）。每个点是一条公司年度配对。
右下方代表库存增长差为正、下一年度比率下降；右上方代表库存增长差为正、下一年度比率上升。
零线用于判断方向，不能当作已经验证的交易阈值。

观察点云是否分散、各公司方向是否一致、是否只有极端点明显。不画自动拟合线，先避免让短小样本看起来有确定规律。
对于同一横坐标分散的正负结果，说明单一库存指标不能完全解释经营变化。
savefig 保存图，show 在 Notebook 展示。图表带来源、样本量和期间范围，便于以后放入报告。
参考：[matplotlib scatter 文档](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html)。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, outcome, title in zip(
    axes,
    ["next_gross_margin_change_pp", "next_ocf_margin_change_pp"],
    ["Next-year gross margin change", "Next-year OCF-to-revenue change"],
):
    for ticker, company in analysis.groupby("ticker"):
        ax.scatter(company["inventory_gap_pp"], company[outcome], label=ticker, alpha=0.8, s=45)
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.set_xlabel("Inventory growth minus sales growth (pp)")
    ax.set_ylabel("Next-year ratio change (pp)")
    ax.set_title(title)
    ax.legend(title="Company")
fig.suptitle("Pilot annual associations")
fig.text(0.5, 0.01, source_caption, ha="center", fontsize=8)
fig.tight_layout(rect=[0, 0.06, 1, 0.93])
fig.savefig(FIGURE_DIR / "pilot_inventory_gap_scatter.png", dpi=180, bbox_inches="tight")
plt.show()


## Cell 5: Compare Positive and Nonpositive Inventory Gaps

先固定一个直观分组规则：Gap > 0 与 Gap <= 0；不根据结果挑选阈值。
正差值表示存货增长率高于收入增长率。收入和存货都下降时也可能出现正差值，它不等于绝对库存增加或已经积压。

分组在每家公司内部进行，不把三家公司混在一起比较。
group_summary 保留各组观察数、平均目标变化和下降比例。柱状图只展示各组平均变化；柱上的 n 是观察数。
.unstack("inventory_group") 把原来的分组行变成两列，便于画并排柱状图。
.reindex 固定两组的展示顺序；如果以后某公司缺少一组，会保留 NaN，不用 0 编造结果。

原假设预期正差值组的下一年变化更低，但实际不一定符合。均值也可能受少数年份影响，要和散点图、样本量一起解读。
这只是样本内描述，不能把历史组内下降比例直接解释为未来概率，也不是两组公司随机试验。


In [ ]:
analysis["inventory_group"] = "Gap <= 0"
analysis.loc[analysis["inventory_sales_gap"] > 0, "inventory_group"] = "Gap > 0"
group_summary = analysis.groupby(["ticker", "inventory_group"]).agg(
    paired_rows=("ticker", "size"),
    mean_next_gross_margin_change_pp=("next_gross_margin_change_pp", "mean"),
    mean_next_ocf_margin_change_pp=("next_ocf_margin_change_pp", "mean"),
    gross_margin_decline_rate=("gross_margin_declined", "mean"),
    ocf_margin_decline_rate=("ocf_margin_declined", "mean"),
)
display(group_summary.round(3))
group_order = ["Gap <= 0", "Gap > 0"]
group_counts = group_summary["paired_rows"].unstack("inventory_group").reindex(columns=group_order)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, metric, title in zip(
    axes,
    ["mean_next_gross_margin_change_pp", "mean_next_ocf_margin_change_pp"],
    ["Mean next-year gross margin change", "Mean next-year OCF-to-revenue change"],
):
    means = group_summary[metric].unstack("inventory_group").reindex(columns=group_order)
    means.plot.bar(ax=ax, rot=0, color=["gray", "tab:blue"])
    for group_name, bars in zip(group_order, ax.containers):
        labels = [f"n={int(n)}" if pd.notna(n) else "" for n in group_counts[group_name]]
        ax.bar_label(bars, labels=labels, padding=3, fontsize=8)
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("Company")
    ax.set_ylabel("Mean next-year ratio change (pp)")
    ax.margins(y=0.2)
    ax.legend(title="Current inventory gap")
fig.text(0.5, 0.01, source_caption, ha="center", fontsize=8)
fig.tight_layout(rect=[0, 0.06, 1, 1])
fig.savefig(FIGURE_DIR / "pilot_inventory_group_means.png", dpi=180, bbox_inches="tight")
plt.show()


## Cell 6: Check Sensitivity to Annual-Period Length Changes

比较两个预先规定的样本：All pairs；排除本期或目标年度出现财年长度变化的 Unchanged period lengths。
使用 Module 5 已保存的 pair_period_length_changed，而不是看到结果后才挑选要删的年份。
完整样本应为 43 行；排除长度变化后应为 27 行，AEO 6、ANF 6、URBN 15。

每家公司、每个目标分别计算 Pearson 相关系数，衡量线性关联方向与强度。
负值表示较高库存差与较低下一年变化同向出现；正值方向相反；接近零不代表不存在所有非线性关系。
只有一条记录或任一列完全不变时，相关系数无法有效计算，保留 NaN，不解释为 0。

保持样本数与系数一起展示。只有 6 条年度记录的结果很容易变化；不计算 p 值，不声称显著性或因果关系。
下一年度长度标记只用于事后审核；不能直接当作本期预测特征。
参考：[pandas Series.corr 文档](https://pandas.pydata.org/docs/reference/api/pandas.Series.corr.html)。


In [ ]:
sensitivity_rows = []
for sample_name, sample in [
    ("All pairs", analysis),
    ("Unchanged period lengths", analysis.loc[~analysis["pair_period_length_changed"]]),
]:
    for ticker, company in sample.groupby("ticker"):
        for outcome in ["next_gross_margin_change_pp", "next_ocf_margin_change_pp"]:
            correlation = float("nan")
            if company["inventory_gap_pp"].nunique() > 1 and company[outcome].nunique() > 1:
                correlation = company["inventory_gap_pp"].corr(company[outcome])
            sensitivity_rows.append({
                "sample": sample_name, "ticker": ticker, "outcome": outcome,
                "paired_rows": len(company), "pearson_r": correlation,
            })
sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity.round(3))


## Cell 7: Select One Case per Company for Filing Review

给每家公司选一个库存增长差最高的年度，形成待核对的个案工作表。
.sort_values(..., ascending=False) 从最大差值开始；.drop_duplicates("ticker") 保留每家公司第一次出现的行。
这里只对个案选取列表去重，不删除完整研究样本。
选择规则依据本期库存差，没有依据下一年度结果挑选；但最大差值本来就是极端观察，因此不能代表典型年份或证明总体规律。

用 ticker 找到 CIK，再用 accn 构造 SEC 文件目录链接。目录通常包含原始年报、提交说明和 XBRL 文件。
current_filing_url 用于解释当时存货与收入变化；next_filing_url 用于核对后续经营表现及可能原因。
本格生成核对链接，不自动下载年报，不替你判断经营原因。

对每个个案，人工查：库存与收入是否下降或上升、是否新增业务或收购、是否有 53 周因素、毛利变化是否由促销/关税/费用分类等解释、经营现金流收入比是否受营运资本时点影响。
未核对前 case_review_status 保持 pending，review_notes 留空。经营现金流收入比下降不等于现金流金额必然下降。


In [ ]:
cases = analysis.sort_values("inventory_sales_gap", ascending=False).drop_duplicates("ticker").copy()
cases["cik"] = cases["ticker"].map({"AEO": 919012, "ANF": 1018840, "URBN": 912615})
for index, row in cases.iterrows():
    cik = int(row["cik"])
    cases.loc[index, "current_filing_url"] = (
        f"https://www.sec.gov/Archives/edgar/data/{cik}/{row['accn'].replace('-', '')}/"
    )
    cases.loc[index, "next_filing_url"] = (
        f"https://www.sec.gov/Archives/edgar/data/{cik}/{row['next_accn'].replace('-', '')}/"
    )
cases["case_review_status"] = "pending"
cases["review_notes"] = ""
case_columns = [
    "ticker", "period_end", "next_period_end", "inventory_gap_pp",
    "next_gross_margin_change_pp", "next_ocf_margin_change_pp",
    "current_filing_url", "next_filing_url", "case_review_status", "review_notes",
]
cases = cases[case_columns]
display(cases)


## Cell 8: Save the Exploratory Outputs and Rules

保存 4 个 CSV 和 Module 6 metadata，两张 PNG 已在画图的 Cell 中保存到 outputs/figures。
字典循环保存表格；reset_index 让公司名与分组名成为普通 CSV 列，避免多层索引保存后不易阅读。
公司和分组表只在保存时转换索引；Notebook 内原表仍保留便于使用的索引。

metadata 保留本模块方法、样本数、信息时间限制和尚未完成的审核。
statistical_inference_performed=False 表示没有做显著性检验，不能把输出误称为“通过检验的投资信号”。
历史口径和个案审核仍未完成。这里只记录探索，下一步再扩展公司样本与完善审核。
再次运行会覆盖 Module 6 同名输出，但不修改 Module 5 配对表。


In [ ]:
outputs = {
    "pilot_company_eda_summary.csv": company_summary.reset_index(),
    "pilot_inventory_group_summary.csv": group_summary.reset_index(),
    "pilot_period_length_sensitivity.csv": sensitivity,
    "pilot_case_review.csv": cases,
}
for filename, data in outputs.items():
    data.to_csv(PROCESSED_DIR / filename, index=False, date_format="%Y-%m-%d")
    print("Saved:", filename)
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    "module5_metadata": module5_metadata,
    "paired_rows": len(analysis),
    "company_count": int(analysis["ticker"].nunique()),
    "group_rule": "Within each company: inventory-sales growth gap > 0 versus <= 0.",
    "sensitivity_rule": "All pairs versus pairs without flagged annual-period length changes.",
    "case_rule": "Highest current inventory-sales growth gap per company; filing review pending.",
    "display_units": "Changes in percentage points; decline rates stored as fractions.",
    "statistical_inference_performed": False,
    "accounting_scope_verified": False,
    "full_year_forecast_ready": False,
}
(PROCESSED_DIR / "pilot_module6_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
print("Saved: pilot_module6_metadata.json")
print("Figure directory:", FIGURE_DIR)


## Completion and Next Step

你应有公司概要、两张探索图、分组描述、两种样本的相关系数及 3 个待审核个案。
这些结果用于形成后续核对问题，不应预先要求相关系数为负或所有公司方向一致。

报告解释可以按以下顺序组织：
1. 明确试点规模与信息时间窗。
2. 描述各家公司点云、分组均值及观察数。
3. 比较财年长度筛选前后的方向和变化。
4. 用年报核对个案，分别解释库存、毛利、现金流。
5. 区分观察到的关联、可能机制和仍缺少的证据。

如果公司之间方向不同或筛选后改变，应如实报告，不直接删掉不符合假设的公司。
下一模块应优先扩展公司样本，并把已识别的标签和历史口径审核规则应用到新增公司。
正式的多公司统计分析应在扩样和审核之后进行；三家公司试点不用于宣称稳定预测能力。
